# 🎬 Movie Recap AI (v2.2) — Google Colab T4 GPU Edition
Welcome to the high-performance Google Colab cloud edition of **Movie Recap AI**! 🚀

### ⚡ Hardware Accelerator (T4 GPU):
Before running, make sure your runtime has GPU enabled:
1. In the menu bar at the top, click: **Runtime** > **Change runtime type**.
2. Under **Hardware accelerator**, select **T4 GPU**.
3. Click **Save**.

> **Note:** Your final recap videos, thumbnails, scripts, and logs are automatically saved permanently in **Google Drive**, while temporary media processing uses Colab's fast local NVMe SSD (`/content/temp`).

### 📦 Step 1: Mount Google Drive, Clone Repo & Install Packages

In [ ]:
# Mount Google Drive
from google.colab import drive
import os

print("[*] Mounting Google Drive...")
drive.mount("/content/drive")

# Project folder in Google Drive
drive_path = "/content/drive/MyDrive/ai-translate-agent"
legacy_path = "/content/drive/MyDrive/AI-Movie-Translate"

if os.path.exists(legacy_path) and not os.path.exists(drive_path):
    drive_path = legacy_path

if not os.path.exists(drive_path):
    print(f"[*] Cloning repository into {drive_path}...")
    %cd /content/drive/MyDrive
    !git clone https://github.com/paipai1999/ai-translate-agent.git
else:
    print(f"[*] Existing project found at {drive_path}! Syncing latest updates...")
    %cd {drive_path}
    !git reset --hard HEAD
    !git pull origin main

%cd {drive_path}

# Setup ultra-fast local NVMe temp directory
os.makedirs("/content/temp", exist_ok=True)

print("[*] Installing dependencies & system fonts (Padauk, Noto, FFmpeg)...")
!pip install -q -r requirements.txt
!apt-get update -qq
!apt-get install -y -qq ffmpeg fonts-sil-padauk fonts-noto-cjk fonts-noto-core

import torch
print(f"\n✅ Setup complete! PyTorch CUDA available: {torch.cuda.is_available()} (Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")

### ⚙️ Step 2: Configure Gemini API Key & Video Settings

In [ ]:
import json
import os
from getpass import getpass

# Enter your Google AI Studio Gemini API Key
# Get your free key at: https://aistudio.google.com/app/apikey
GEMINI_API_KEY = ""  # Or paste directly here: e.g. "AIzaSy..."

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass("Enter your Gemini API Key (hidden for security): ").strip()

config_data = {
    "gemini": {
        "enabled": True,
        "api_keys": [GEMINI_API_KEY] if GEMINI_API_KEY else [],
        "model": "gemini-3.5-flash-lite",
        "daily_limit_per_key": 20,
        "model_limits": {
            "gemini-3.5-flash-lite": 20,
            "gemini-3.6-flash": 15,
            "gemini-2.5-flash": 10,
            "gemini-2.0-flash": 15
        },
        "models": {
            "heavy": "gemini-3.5-flash-lite",
            "workhorse": "gemini-3.5-flash-lite",
            "polish": "gemini-3.5-flash-lite"
        }
    },
    "pipeline": {
        "language": "burmese",
        "whisper_model": "small",
        "scene_threshold": 30.0,
        "max_characters": 6,
        "max_scenes_for_llm": 30,
        "parallel_processing": True,
        "use_demucs": True
    },
    "voice": {
        "enabled": True,
        "engine": "edge_tts",
        "tts_voice_mm": "my-MM-ThihaNeural",
        "tts_voice_en": "en-US-GuyNeural",
        "tts_voice": "my-MM-ThihaNeural",
        "tts_rate_mm": "+8%",
        "tts_rate_en": "+15%",
        "f5_tts": {
            "model_type": "F5-TTS",
            "auto_character_cloning": True,
            "default_ref_audio": "assets/voices/default_ref.wav",
            "default_ref_text": "",
            "speed": 1.0,
            "device": "auto"
        }
    },
    "batch": {
        "movies_folder": "movies",
        "max_parallel_jobs": 1,
        "skip_completed": True
    },
    "paths": {
        "temp_dir": "/content/temp",
        "output_dir": "outputs"
    },
    "copyright_protection": {
        "enabled": True,
        "mirror_video": False,
        "resize_factor": 1.02
    },
    "subtitle_blur": {
        "enabled": True,
        "region_height_pct": 0.18,
        "blur_strength": 18
    },
    "color_grading": {
        "enabled": True,
        "brightness": 0.03,
        "contrast": 1.02,
        "saturation": 1.08
    },
    "subtitle_overlay": {
        "enabled": True,
        "font_name": "Padauk",
        "font_size": 40,
        "bold": True,
        "border_style": 3,
        "outline_width": 3,
        "margin_bottom": 50,
        "max_chars_per_line": 28
    },
    "logging": {
        "level": "INFO",
        "save_log_file": True
    }
}

with open("config.json", "w", encoding="utf-8") as f:
    json.dump(config_data, f, indent=4)

print("\n✅ config.json saved successfully in Google Drive!")
print(f"🚀 Model: {config_data['gemini']['model']} | Language: {config_data['pipeline']['language']} | Voice Engine: {config_data['voice']['engine']}")

### 🚀 Step 3 (Option A): Run via Command Line (Fastest & Simplest)
Paste your YouTube URL (or local video path in `movies/`) and click Run:

In [ ]:
# Input Video URL or local path
VIDEO_URL = "https://www.youtube.com/watch?v=5VRSIZwxJso"  # <--- Paste your video link here
LANGUAGE = "burmese"                                     # "burmese" or "english"
ENGINE = "edge_tts"                                      # "edge_tts" (Fast $0) or "f5_tts" (Voice Clone)

# Run Autonomous AI Recap Pipeline
!python main.py "$VIDEO_URL" -l "$LANGUAGE" -e "$ENGINE"

### 🌐 Step 3 (Option B): Launch Modern Web Dashboard UI (Interactive)
If you prefer the visual browser dashboard with live progress and real-time streaming console, run this cell:

In [ ]:
# Download cloudflared tunnel client if not already present
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import os, subprocess, time, sys

# Clean up any lingering processes
!pkill -f "web_ui.py" || true
!pkill -f "cloudflared" || true

print("[*] Starting Web Dashboard Server...")
web_proc = subprocess.Popen(
    ["python", "web_ui.py"],
    stdout=open("/content/web_ui.log", "w"),
    stderr=subprocess.STDOUT
)
time.sleep(3)

if web_proc.poll() is not None:
    print("❌ ERROR: Web UI failed to start! Check logs below:")
    with open("/content/web_ui.log", "r") as f:
        print(f.read())
else:
    print("\n===========================================================")
    print("🌐 CREATING SECURE CLOUDFLARE PUBLIC TUNNEL...")
    print("👇 Click the URL ending with '.trycloudflare.com' below 👇")
    print("===========================================================\n")
    !./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:5000

### 📁 Step 4: Check Generated Outputs
All output recap videos, thumbnails, scripts, and logs are automatically preserved in your Google Drive.

In [ ]:
import os

print("\n📦 COMPLETED OUTPUTS IN GOOGLE DRIVE:")
outputs_dir = "outputs"
if os.path.exists(outputs_dir):
    for movie in sorted(os.listdir(outputs_dir)):
        p = os.path.join(outputs_dir, movie)
        if os.path.isdir(p):
            print(f"\n🎥 {movie}/")
            for f in sorted(os.listdir(p)):
                size_mb = os.path.getsize(os.path.join(p, f)) / (1024 * 1024)
                print(f"   ├─ {f:<30} ({size_mb:.2f} MB)")
else:
    print("No outputs yet.")